## 01b_ingest_static_files — RCM Intelligence Platform
### Purpose
Downloads the CMS Hospital General Information static CSV file
and lands it into the Bronze layer as a Delta table.

### Data sources
- CMS Hospital General Information (Provider Compare static CSV)
  Contains: hospital name, type, ownership, address, quality ratings
  for 5,000+ Medicare-certified hospitals across the US

### What this does
1. Downloads CSV directly from CMS Provider Data URL
2. Saves raw file to Unity Catalog Volume
3. Reads into Spark DataFrame
4. Adds audit metadata columns
5. Writes to Bronze Delta table
6. Logs run to audit table

### Outputs
- rcm_platform.rcm_bronze.hospital_info_raw

### Notes
- Static file — CMS updates this quarterly
- URL is stable — does not rotate like API UUIDs
- Safe to re-run — each run gets a unique batch ID

| Field      | Value |
|------------|-------|
| Author     | Mayank Joshi |
| Layer      | Bronze |
| Run order  | 3 — after 01a_ingest_cms_api |
| Depends on | 00_config, 00_utils |

In [0]:
%run /Users/jmayank574@gmail.com/rcm-intelligence-platform/00_setup/00_config

In [0]:
%run /Users/jmayank574@gmail.com/rcm-intelligence-platform/00_setup/00_utils

In [0]:
# ============================================================
# BATCH METADATA
# ============================================================

import uuid
import requests
from datetime import datetime, timezone

BATCH_ID        = str(uuid.uuid4())
BATCH_DATE      = datetime.now(timezone.utc).strftime("%Y-%m-%d")
BATCH_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
NOTEBOOK_NAME   = "01b_ingest_static_files"

print(f"Batch ID        : {BATCH_ID}")
print(f"Batch date      : {BATCH_DATE}")
print(f"Batch timestamp : {BATCH_TIMESTAMP}")

In [0]:
# ============================================================
# STEP 1 — DOWNLOAD HOSPITAL INFO CSV TO VOLUME
# ============================================================

print("=" * 55)
print("  DOWNLOADING HOSPITAL GENERAL INFORMATION CSV")
print("=" * 55)

hospital_file_path = f"{RAW_HOSPITAL_PATH}/hospital_info_{BATCH_DATE}.csv"

print(f"Source : {CMS_HOSPITAL_INFO_CSV}")
print(f"Target : {hospital_file_path}")
print(f"\nDownloading...")

response = requests.get(
    CMS_HOSPITAL_INFO_CSV,
    timeout=60,
    allow_redirects=True
)
response.raise_for_status()

# Verify we got actual CSV content not an HTML page
first_line = response.text.strip().split("\n")[0]
if "<html" in first_line.lower():
    raise ValueError(
        "Got HTML instead of CSV — URL may have redirected to a webpage"
    )

print(f"Content received : {len(response.content):,} bytes")
print(f"First line       : {first_line[:80]}")

# Write to Unity Catalog Volume
dbutils.fs.put(
    hospital_file_path,
    response.text,
    overwrite=True
)

print(f"\nDownload complete")
print(f"File saved to: {hospital_file_path}")

In [0]:
# ============================================================
# STEP 2 — READ CSV INTO SPARK DATAFRAME
# ============================================================

print("=" * 55)
print("  READING CSV INTO SPARK")
print("=" * 55)

df_hospital_raw = spark.read \
    .option("header",      "true") \
    .option("inferSchema", "false") \
    .option("multiLine",   "true") \
    .option("escape",      '"') \
    .option("quote",       '"') \
    .option("sep",         ",") \
    .csv(hospital_file_path)

raw_count = df_hospital_raw.count()

print(f"\nRows read : {raw_count:,}")
print(f"Columns   : {len(df_hospital_raw.columns)}")
print(f"\nColumn names:")
for c in df_hospital_raw.columns:
    print(f"  {c}")

In [0]:
# ============================================================
# STEP 3 — CLEAN COLUMN NAMES + ADD AUDIT METADATA + WRITE
# Delta Lake does not allow spaces or special chars in column names
# We clean them here — Bronze stores raw values, clean names
# ============================================================

print("=" * 55)
print("  WRITING TO BRONZE DELTA TABLE")
print("=" * 55)

from pyspark.sql.functions import lit, col
from pyspark.sql.types import StringType
import re

def clean_column_name(name: str) -> str:
    """
    Makes a column name Delta-safe:
    - Lowercase
    - Spaces and special chars replaced with underscore
    - Multiple underscores collapsed to one
    - Leading/trailing underscores removed
    """
    name = name.lower()
    name = re.sub(r"[^a-z0-9]", "_", name)
    name = re.sub(r"_+", "_", name)
    name = name.strip("_")
    return name

# Cast all columns to string and clean names
df_hospital_clean = df_hospital_raw.select(
    [
        col(c).cast(StringType()).alias(clean_column_name(c))
        for c in df_hospital_raw.columns
    ]
)

print("Cleaned column names:")
for c in df_hospital_clean.columns:
    print(f"  {c}")

# Add audit metadata
df_hospital_bronze = df_hospital_clean \
    .withColumn("_source",           lit("cms_hospital_compare_csv")) \
    .withColumn("_source_file",      lit(hospital_file_path)) \
    .withColumn("_batch_id",         lit(BATCH_ID)) \
    .withColumn("_batch_date",       lit(BATCH_DATE)) \
    .withColumn("_ingested_at",      lit(BATCH_TIMESTAMP)) \
    .withColumn("_pipeline_name",    lit(PIPELINE_NAME)) \
    .withColumn("_pipeline_version", lit(PIPELINE_VERSION))

hospital_rows = write_delta(
    df         = df_hospital_bronze,
    table_name = TBL_BRONZE_HOSPITAL_RAW,
    mode       = "overwrite",
    comment    = (
        "Raw CMS Hospital General Information — provider name, type, "
        "ownership, address and quality ratings for all Medicare-certified hospitals"
    )
)

print(f"\nBronze table : {TBL_BRONZE_HOSPITAL_RAW}")
print(f"Rows written : {hospital_rows:,}")

print("\nSample rows:")
display(
    df_hospital_bronze.select(
        "facility_id",
        "facility_name",
        "hospital_type",
        "hospital_ownership",
        "state",
        "city_town",
        "emergency_services",
        "hospital_overall_rating"
    ).limit(10)
)

In [0]:
# ============================================================
# STEP 4 — VERIFICATION
# ============================================================

print("=" * 55)
print("  BRONZE VERIFICATION — HOSPITAL INFO")
print("=" * 55)

spark.sql(f"""
    SELECT
        COUNT(*)                          AS total_hospitals,
        COUNT(DISTINCT state)             AS states_covered,
        COUNT(DISTINCT hospital_type)     AS hospital_types,
        COUNT(DISTINCT hospital_ownership) AS ownership_types,
        SUM(CASE WHEN emergency_services = 'Yes'
            THEN 1 ELSE 0 END)            AS with_emergency_services,
        SUM(CASE WHEN hospital_overall_rating IS NOT NULL
            AND hospital_overall_rating != 'Not Available'
            THEN 1 ELSE 0 END)            AS with_quality_rating
    FROM {TBL_BRONZE_HOSPITAL_RAW}
""").show(truncate=False)

print("Hospital types breakdown:")
spark.sql(f"""
    SELECT
        hospital_type,
        COUNT(*) AS hospital_count
    FROM {TBL_BRONZE_HOSPITAL_RAW}
    GROUP BY hospital_type
    ORDER BY hospital_count DESC
""").show(truncate=False)

print("Top 10 states by hospital count:")
spark.sql(f"""
    SELECT
        state,
        COUNT(*) AS hospital_count
    FROM {TBL_BRONZE_HOSPITAL_RAW}
    GROUP BY state
    ORDER BY hospital_count DESC
    LIMIT 10
""").show(truncate=False)

print("Delta history:")
spark.sql(f"""
    DESCRIBE HISTORY {TBL_BRONZE_HOSPITAL_RAW} LIMIT 3
""").select("version", "timestamp", "operation", "operationMetrics") \
    .show(truncate=False)

In [0]:
# ============================================================
# STEP 5 — LOG TO AUDIT TABLE
# ============================================================

log_pipeline_run(
    notebook_name    = NOTEBOOK_NAME,
    layer            = "bronze",
    status           = "SUCCESS",
    rows_read        = hospital_rows,
    rows_written     = hospital_rows,
    rows_quarantined = 0,
    message          = (
        f"Batch {BATCH_ID} — "
        f"hospital info: {hospital_rows:,} rows | "
        f"38 columns | snake_case names applied | "
        f"source: xubh-q36u"
    )
)